# Speech Recognition

Companion notebook for the [Speech Recognition lesson](https://ml-viz-ruby.vercel.app/courses/speech-audio/02-speech-recognition).

**The problem in one sentence.** A model emits one character-distribution *per
audio frame* (say 100/second), but the transcript has far fewer characters and
you have **no idea which frame produced which letter** — CTC solves this
alignment problem without ever needing frame-level labels.

**The trick — the blank + collapse rule.** Allow a special blank token `_`, then
*collapse* each frame-labeling by merging repeats and dropping blanks. Many
frame-labelings collapse to the same text, so CTC defines the probability of a
transcript as the **sum over all alignments that collapse to it** — computed with
dynamic programming, not enumeration.

Roadmap: the collapse rule → counting alignments → the **CTC forward algorithm**
(which we **validate against brute-force enumeration**) → greedy decoding → **WER**.
Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
BLANK = '_'    # the CTC blank token

## 1 — The CTC collapse rule

Collapse a frame-level labeling to text: first merge consecutive repeats, then remove blanks. The
blank is what lets 'hello' (with a real double-l) survive.

In [ ]:
def ctc_collapse(path):
    out = []
    prev = None
    for c in path:
        if c != prev:        # merge consecutive repeats
            out.append(c)
        prev = c
    return ''.join(c for c in out if c != BLANK)   # then drop blanks

for path in ['hhh_e_lll_llo', 'h_e_l_l_o', 'hello']:
    print(f'{path:15s} -> "{ctc_collapse(path)}"')
print('\nNote the blank between the two l\'s is what preserves the double-l in "hello".')

## 2 — Many alignments map to one target

CTC sums the probability over *every* frame-labeling that collapses to the target. We enumerate the
alignments of a short target over a few frames to see how many there are.

In [ ]:
from itertools import product

def count_alignments(target, T, vocab):
    n = 0
    for path in product(vocab, repeat=T):
        if ctc_collapse(path) == target:
            n += 1
    return n

vocab = ['a', 'b', BLANK]
for T in [3, 4, 5, 6]:
    print(f'T={T} frames: {count_alignments("ab", T, vocab)} alignments collapse to "ab"')
print('\nThe count grows fast -> CTC sums them with dynamic programming, never enumerating.')

## The CTC forward algorithm — sum over alignments in $O(T\cdot S)$

The alignment count exploded above, so CTC never enumerates. Instead it runs a
**forward dynamic program** over the *blank-extended* label sequence
$\_\,l_1\,\_\,l_2\,\_\,\dots$ (a blank between and around every target character).
$\alpha_t(s)$ = total probability of all paths that have produced the first $s$
symbols of the extended sequence by frame $t$. Each state can come from itself,
the previous state, or (skipping a blank) two states back:

$$\alpha_t(s) = p_t(\text{ext}[s]) \cdot \big(\alpha_{t-1}(s) + \alpha_{t-1}(s-1) + [\,\text{skip allowed}\,]\,\alpha_{t-1}(s-2)\big)$$

The total transcript probability is $\alpha_T(S) + \alpha_T(S-1)$.

In [ ]:
def ctc_forward(probs, target, vocab, blank=BLANK):
    '''Total probability of `target` under CTC, summed over all alignments.

    probs: (T, V) per-frame token distribution. Returns a scalar.'''
    b = vocab.index(blank)
    ext = [b]
    for ch in target:
        ext += [vocab.index(ch), b]        # blank between/around every char
    T, Sx = len(probs), len(ext)
    a = np.zeros((T, Sx))
    a[0, 0] = probs[0, ext[0]]
    a[0, 1] = probs[0, ext[1]]
    for t in range(1, T):
        for s in range(Sx):
            val = a[t-1, s]
            if s > 0:
                val += a[t-1, s-1]
            # skip a blank only if the current symbol is a non-blank distinct from ext[s-2]
            if s > 1 and ext[s] != b and ext[s] != ext[s-2]:
                val += a[t-1, s-2]
            a[t, s] = val * probs[t, ext[s]]
    return a[T-1, Sx-1] + a[T-1, Sx-2]

# quick run on a toy distribution
vocab2 = ['c', 'a', 't', BLANK]
rng2 = np.random.default_rng(1)
probs_toy = rng2.random((5, 4)); probs_toy /= probs_toy.sum(1, keepdims=True)
print(f'P("cat") via forward DP: {ctc_forward(probs_toy, "cat", vocab2):.6f}')

### Validate: the DP equals the brute-force sum over alignments

The forward algorithm is only correct if $\alpha_T$ equals the *explicit* sum of
per-path probabilities over every frame-labeling that collapses to the target.
For small $T$ we can enumerate all $V^T$ paths and check — the DP must match to
machine precision.

In [ ]:
from itertools import product

def brute_force_prob(probs, target, vocab):
    total = 0.0
    for path in product(range(len(vocab)), repeat=len(probs)):
        if ctc_collapse(''.join(vocab[i] for i in path)) == target:
            p = np.prod([probs[t, i] for t, i in enumerate(path)])
            total += p
    return total

for T in [4, 5, 6]:
    pr = np.random.default_rng(T).random((T, 4)); pr /= pr.sum(1, keepdims=True)
    dp = ctc_forward(pr, 'cat', vocab2)
    bf = brute_force_prob(pr, 'cat', vocab2)
    print(f'T={T}: forward DP = {dp:.6f}   brute force = {bf:.6f}   match = {np.isclose(dp, bf)}')
    assert np.isclose(dp, bf), 'forward DP must equal the exact sum over alignments'
print('\n✅ the O(T·S) forward algorithm exactly reproduces the sum over all alignments')

## 3 — Greedy CTC decoding

The simplest decode: at each frame take the most likely token, then collapse. We make a toy
per-frame probability matrix and decode it.

In [ ]:
vocab2 = ['c', 'a', 't', BLANK]
# per-frame probabilities (rows = frames, cols = vocab) peaking at c,a,a,t,blank,t
frames = ['c', 'a', 'a', 't', BLANK, 't']
probs = np.full((len(frames), len(vocab2)), 0.05)
for i, ch in enumerate(frames):
    probs[i, vocab2.index(ch)] = 0.85

def greedy_decode(probs, vocab):
    path = ''.join(vocab[i] for i in probs.argmax(axis=1))
    return ctc_collapse(path), path

text, path = greedy_decode(probs, vocab2)
print(f'argmax path: {path}')
print(f'decoded:     "{text}"')

**What to notice — greedy decoding.** Taking the per-frame argmax and collapsing
is fast but *not* optimal: it ignores that many low-probability paths can sum to a
transcript more likely than the single best path. That's why production ASR uses
**beam search** over the CTC lattice (often with a language model). Greedy is the
baseline; the forward algorithm above is what training actually optimises.

## Word Error Rate (WER) — the standard ASR metric

Accuracy is meaningless when the hypothesis and reference have different lengths,
so ASR is scored with **WER**: the word-level edit distance (substitutions +
insertions + deletions) divided by the number of reference words.

$$\text{WER} = \frac{S + I + D}{N_{\text{ref}}}$$

In [ ]:
def wer(reference, hypothesis):
    ref, hyp = reference.split(), hypothesis.split()
    # Levenshtein edit distance at the word level
    d = np.zeros((len(ref)+1, len(hyp)+1), dtype=int)
    d[:, 0] = np.arange(len(ref)+1)
    d[0, :] = np.arange(len(hyp)+1)
    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i, j] = min(d[i-1, j] + 1,        # deletion
                          d[i, j-1] + 1,        # insertion
                          d[i-1, j-1] + cost)   # substitution / match
    return d[len(ref), len(hyp)] / max(len(ref), 1)

tests = [
    ('the cat sat on the mat', 'the cat sat on the mat'),   # perfect
    ('the cat sat on the mat', 'the cat sat on a mat'),      # 1 substitution
    ('the cat sat on the mat', 'the cat on the mat'),        # 1 deletion
    ('the cat sat',            'the big cat sat quietly'),   # insertions
]
for ref, hyp in tests:
    print(f'WER={wer(ref, hyp):.2f}  |  ref: "{ref}"  hyp: "{hyp}"')
assert wer('a b c', 'a b c') == 0.0
assert wer('a b c d', 'a x c d') == 0.25           # 1 sub / 4 words
print('\n✅ WER measures word-level edit distance normalised by reference length')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **the blank is essential** | without `_`, a real double letter (`hello`) collapses to `helo`; the blank separates repeat groups |
| **greedy ≠ optimal** | the single best path can lose to a transcript whose many paths sum higher → use beam search |
| **CTC's independence assumption** | it factorises per frame, so it has no built-in language model — pair it with one for accuracy |
| **WER can exceed 100%** | many insertions on a short reference; it's a ratio, not a percentage of "words gotten right" |
| **frame rate vs label length** | need $T \ge$ (chars + repeats); too few frames makes some transcripts impossible |

Demo: the blank really is load-bearing — dropping it breaks any word with a
repeated letter.

In [ ]:
# WITH a separating blank between the two l-groups, the double-l survives:
print(f'"hh_e_ll_llo"  -> "{ctc_collapse("hh_e_ll_llo")}"  (blank preserves the double-l)')
# WITHOUT a blank between them, the repeated l's merge and the word is corrupted:
print(f'"hhelllo"      -> "{ctc_collapse("hhelllo")}"  (no separating blank -> double-l lost)')
assert ctc_collapse('hh_e_ll_llo') == 'hello'
assert ctc_collapse('hhelllo') == 'helo'
print('\nThe blank between repeat groups is the ONLY reason CTC can emit double letters.')

## ✏️ Your turn

**Exercise.** Implement `collapse(path, blank)` (the general CTC collapse: merge repeats then drop
the blank symbol) and `is_valid_alignment(path, target, blank)` (True if the path collapses exactly
to the target). These define what CTC sums over.

In [ ]:
def collapse(path, blank='_'):
    # TODO(you): merge consecutive duplicates, then remove the blank token
    return ...

def is_valid_alignment(path, target, blank='_'):
    # TODO(you): True if collapse(path) equals target
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert collapse('hh_e_ll_llo') == 'hello'        # blank preserves the double-l
assert collapse('aa_aa') == 'aa'                  # blank separates two a-groups
assert collapse('aaaa') == 'a'                    # no blank -> repeats merge to one
assert is_valid_alignment('c_aa_t', 'cat')
assert not is_valid_alignment('caat', 'caat')    # collapses to 'cat', not 'caat'
print('\u2713 CTC collapse and alignment check are correct')

<details>
<summary>Solution</summary>

```python
def collapse(path, blank='_'):
    out, prev = [], None
    for c in path:
        if c != prev:
            out.append(c)
        prev = c
    return ''.join(c for c in out if c != blank)

def is_valid_alignment(path, target, blank='_'):
    return collapse(path, blank) == target
```

The collapse rule is the whole reason CTC needs no frame-level labels: the model can emit any
alignment that collapses to the target, and training sums their probabilities via dynamic
programming.

</details>

## Key takeaways

- **CTC solves alignment without frame labels.** A blank token + collapse rule
  (merge repeats, drop blanks) lets many frame-labelings map to one transcript.
- **The transcript probability is a sum over alignments**, computed by the
  **forward DP** in $O(T\cdot S)$ — we verified it equals the brute-force sum
  over all collapsing paths exactly.
- **The blank is load-bearing:** it's the only way a repeated letter survives
  collapse (`hello`, not `helo`).
- **Greedy decode is a baseline, not optimal** — beam search over the lattice
  (plus a language model, since CTC assumes per-frame independence) wins.
- **ASR is scored with WER** = word-level edit distance / reference length — the
  right metric when hypothesis and reference differ in length.